In [47]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.integrate as spi
from scipy.interpolate import interp1d, RegularGridInterpolator
import pandas as pd
import os
import csv
import warnings
from tqdm import tqdm

# Silencia avisos numéricos esperados nos extremos das integrais
warnings.filterwarnings("ignore", category=spi.IntegrationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Constantes Físicas Globais
c = 2.998e10                   
e = 4.803e-10                  
m_e = 9.109e-28                
m_p = 1.6726e-24               
h = 6.626e-27                  
gev_to_erg = 1.60218e-3       
eV_to_erg = 1.60218e-12       
mec2_erg = m_e * c**2          
sigma_T = 6.65e-25             
d_CenA_cm = 3.8 * 3.086e24 # Distância até Cen A (Ajuste se for rodar outras fontes)

# FÍSICA DE PÍONS E MÚONS
m_pi_GeV = 0.1396
m_pi_erg = m_pi_GeV * gev_to_erg
tau_0_pi = 2.6e-8 

m_mu_GeV = 0.10566
m_mu_erg = m_mu_GeV * gev_to_erg
r_pi = (m_mu_GeV / 0.1396)**2  
tau_0_mu = 2.2e-6 

def T_decay_pion(E_pi_prime_erg):
    return (E_pi_prime_erg / m_pi_erg) * tau_0_pi

def T_decay_muon(E_mu_prime_erg):
    return (E_mu_prime_erg / m_mu_erg) * tau_0_mu

In [48]:
# Mantenha "data/" se os fótons e npy dos prótons estiverem lá dentro.
# Se tudo estiver na raiz junto com o arquivo de parâmetros, mude para ""
DATA_DIR = "data/" 

def load_base_params(source_name, filepath="all_calculated_parameters.csv"):
    """Carrega os parâmetros da fonte a partir do CSV principal na raiz."""
    df = pd.read_csv(filepath)
    source_data = df[df['Source_Name'] == source_name]
    if source_data.empty:
        raise ValueError(f"Fonte {source_name} não encontrada no arquivo {filepath}.")
    
    base_p = source_data.iloc[0].to_dict()
    for k, v in base_p.items():
        if k != 'Source_Name':
            try:
                base_p[k] = float(v)
            except:
                pass
    return base_p

def reconstruct_photon_interp(source_name):
    """Reconstrói a interpolação usando o nome com espaço original."""
    df = pd.read_csv(f"{DATA_DIR}{source_name}_photons.csv")
    E_ph_erg = df['E_ph_erg'].values
    n_ph = df['n_ph_density'].values
    
    n_ph_safe = np.where(n_ph > 1e-100, n_ph, 1e-100)
    interp_log = interp1d(np.log10(E_ph_erg), np.log10(n_ph_safe), 
                          kind='linear', fill_value=-100.0, bounds_error=False)
    
    def n_ph_interp_obj(E):
        if E <= 0: return 0.0
        log_n = interp_log(np.log10(E))
        return 10**log_n if log_n > -100 else 0.0
    return n_ph_interp_obj

def reconstruct_proton_interp(source_name):
    """Carrega as matrizes do NumPy usando o nome com espaço original."""
    N_grid_pro = np.load(f"{DATA_DIR}{source_name}_protons_N_grid.npy")
    E_grid_pro = np.load(f"{DATA_DIR}{source_name}_protons_E_grid.npy")
    z_grid = np.load(f"{DATA_DIR}{source_name}_protons_z_grid.npy")
    
    log_E = np.log10(E_grid_pro)
    log_z = np.log10(z_grid)
    log_N = np.log10(np.where(N_grid_pro > 1e-300, N_grid_pro, 1e-300))
    
    interp_2d = RegularGridInterpolator((log_E, log_z), log_N, method='linear', bounds_error=False, fill_value=-300.0)
    E_max_pro_erg = E_grid_pro[-1]
    
    def N_pro_interp_2d(E_erg, z_cm):
        if E_erg > E_max_pro_erg * 1.5: return 0.0
        pt = np.array([[np.log10(float(E_erg)), np.log10(float(z_cm))]])
        return float(10.0 ** interp_2d(pt)[0])

    return N_pro_interp_2d, E_max_pro_erg

In [49]:
def load_base_params(source_name, filepath="all_calculated_parameters.csv"):
    """Carrega os parâmetros da fonte a partir do CSV principal."""
    df = pd.read_csv(filepath)
    source_data = df[df['Source_Name'] == source_name]
    if source_data.empty:
        raise ValueError(f"Fonte {source_name} não encontrada no arquivo {filepath}.")
    
    # Converte a linha de volta para um dicionário
    base_p = source_data.iloc[0].to_dict()
    # Limpa as strings que deveriam ser numéricas (exceto o nome)
    for k, v in base_p.items():
        if k != 'Source_Name':
            try:
                base_p[k] = float(v)
            except:
                pass
    return base_p

def reconstruct_photon_interp(source_name):
    """Reconstrói a interpolação do campo de fótons (Sync + IC)."""
    df = pd.read_csv(f"data/{source_name}_photons.csv")
    E_ph_erg = df['E_ph_erg'].values
    n_ph = df['n_ph_density'].values
    
    # Previne log(0)
    n_ph_safe = np.where(n_ph > 1e-100, n_ph, 1e-100)
    
    interp_log = interp1d(np.log10(E_ph_erg), np.log10(n_ph_safe), 
                          kind='linear', fill_value=-100.0, bounds_error=False)
    
    def n_ph_interp_obj(E):
        if E <= 0: return 0.0
        log_n = interp_log(np.log10(E))
        return 10**log_n if log_n > -100 else 0.0
    return n_ph_interp_obj

def reconstruct_proton_interp(source_name):
    """Carrega as matrizes do NumPy e reconstrói o N_pro_interp_2d."""
    N_grid_pro = np.load(f"data/{source_name}_protons_N_grid.npy")
    E_grid_pro = np.load(f"data/{source_name}_protons_E_grid.npy")
    z_grid = np.load(f"data/{source_name}_protons_z_grid.npy")
    
    log_E = np.log10(E_grid_pro)
    log_z = np.log10(z_grid)
    log_N = np.log10(np.where(N_grid_pro > 1e-300, N_grid_pro, 1e-300))
    
    interp_2d = RegularGridInterpolator((log_E, log_z), log_N, method='linear', bounds_error=False, fill_value=-300.0)
    
    # Extrai o E_max para reproduzir o corte exponencial
    E_max_pro_erg = E_grid_pro[-1] # Simplificação segura baseada na construção original
    
    def N_pro_interp_2d(E_erg, z_cm):
        if E_erg > E_max_pro_erg * 1.5: return 0.0
        pt = np.array([[np.log10(float(E_erg)), np.log10(float(z_cm))]])
        val = float(10.0 ** interp_2d(pt)[0])
        return val

    return N_pro_interp_2d, E_max_pro_erg

In [50]:
# Funções de Cinemática e Geometria Auxiliares (Necessárias para o Resfriamento)
def calculate_B_field(B_0_val, Z_pos, Z_launch, m):
    if Z_pos <= 0: return 0.0
    return B_0_val * (Z_launch / Z_pos)**m

def calculate_B_prime(B_val, Gamma_b):
    return np.sqrt(3 / (2 * Gamma_b**2 + 1)) * B_val

def t_sync_rate(m_e, B_prime, E_prime, c, sigma_T):
    return (4.0/3.0) * (sigma_T * B_prime**2 * E_prime) / (m_e**2 * c**3 * 8 * np.pi)

def t_adiabatic_rate(Z_pos, beta, c):
    v_bulk = beta * c
    return (2.0/3.0) * (v_bulk / Z_pos)

# =========================================================================
# KELNER & AHARONIAN 2008 - EXATO (p-gamma)
# =========================================================================
_eta_grid_kelner = np.array([1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 30.0, 100.0])
_B_numu_bar = np.array([0.809, 7.70, 19.9, 36.2, 53.9, 73.9, 94.8, 120, 147, 174, 338, 517, 761, 957, 1110, 1250, 1360, 1460, 5870, 31000])
_s_numu_bar = np.array([0.365, 0.287, 0.250, 0.238, 0.220, 0.206, 0.197, 0.193, 0.187, 0.178, 0.123, 0.106, 0.0944, 0.0829, 0.0801, 0.0752, 0.0680, 0.0615, 0.0361, 0.0228])
_del_numu_bar = np.array([3.09, 2.96, 2.89, 2.76, 2.71, 2.67, 2.62, 2.56, 2.52, 2.51, 2.48, 2.56, 2.57, 2.58, 2.54, 2.53, 2.56, 2.60, 2.78, 2.88])
_B_numu = np.array([1.08, 9.91, 24.7, 44.3, 67.0, 90.4, 118, 132, 177, 211, 383, 509, 726, 926, 1070, 1190, 1290, 1400, 5650, 30100])
_s_numu = np.array([0.0, 0.0778, 0.242, 0.377, 0.440, 0.450, 0.461, 0.451, 0.464, 0.446, 0.366, 0.249, 0.204, 0.174, 0.156, 0.140, 0.121, 0.107, 0.0705, 0.0463])
_del_numu = np.array([0.0, 0.306, 0.792, 1.09, 1.06, 0.953, 0.956, 0.922, 0.912, 0.940, 1.49, 2.03, 2.18, 2.24, 2.28, 2.32, 2.39, 2.46, 2.53, 2.62])

_interp_log_B_bar = interp1d(np.log10(_eta_grid_kelner), np.log10(_B_numu_bar), fill_value='extrapolate')
_interp_s_bar     = interp1d(np.log10(_eta_grid_kelner), _s_numu_bar, fill_value='extrapolate')
_interp_del_bar   = interp1d(np.log10(_eta_grid_kelner), _del_numu_bar, fill_value='extrapolate')
_interp_log_B_nu  = interp1d(np.log10(_eta_grid_kelner), np.log10(_B_numu), fill_value='extrapolate')
_interp_s_nu      = interp1d(np.log10(_eta_grid_kelner), _s_numu, fill_value='extrapolate')
_interp_del_nu    = interp1d(np.log10(_eta_grid_kelner), _del_numu, fill_value='extrapolate')

def get_kelner_params(rho, lepton_type):
    if rho < 1.1: return 0.0, 0.0, 0.0
    rho_safe = min(rho, 100.0)
    log_rho = np.log10(rho_safe)
    fator_conversao = 1e-30 * 2.9979e10 
    
    if lepton_type == 'numu_bar':
        return (10**_interp_log_B_bar(log_rho)) * fator_conversao, _interp_s_bar(log_rho), _interp_del_bar(log_rho)
    else:
        return (10**_interp_log_B_nu(log_rho)) * fator_conversao, _interp_s_nu(log_rho), _interp_del_nu(log_rho)

def Phi_nu_Kelner(eta, x, lepton_type):
    eta0, r = 0.313, 0.146 
    if eta < eta0: return 0.0
    rho = eta / eta0
    
    term_sqrt = np.sqrt(max(0, (eta - r**2 - 2*r) * (eta - r**2 + 2*r)))
    x_minus = (eta + r**2 - term_sqrt) / (2 * (1.0 + eta))
    x_plus  = (eta + r**2 + term_sqrt) / (2 * (1.0 + eta))
    psi = 2.5 + 1.4 * np.log(rho)
    
    if lepton_type == 'numu_bar':
        x_min_prime, x_max_prime = x_minus / 4.0, x_plus
    else:
        x_min_prime = 0.427 * x_minus
        if rho < 2.14: x_max_prime = 0.427 * x_plus
        elif rho < 10.0: x_max_prime = (0.427 + 0.0729 * (rho - 2.14)) * x_plus
        else: x_max_prime = x_plus
            
    B, s, delta = get_kelner_params(rho, lepton_type)
    
    if x < x_min_prime: return B * (np.log(2.0))**psi
    elif x_min_prime <= x < x_max_prime:
        y_prime = (x - x_min_prime) / (x_max_prime - x_min_prime)
        return B * np.exp(-s * (np.log(x / x_min_prime))**delta) * (np.log(2.0 / (1.0 + y_prime**2)))**psi
    else: return 0.0

def calculate_Q_nu_pg_exact(E_nu_erg, z_cm, N_pro_interp_2d, n_ph_interp_obj, base_p, E_max_pro):
    x_min = E_nu_erg / E_max_pro
    if x_min >= 1.0: return 0.0
    
    x_grid = np.logspace(np.log10(x_min), 0, 70)
    integral_x = []
    fator_diluicao_foton = (base_p['Z_acc_cm'] / z_cm)**2
    
    for x in x_grid:
        E_p_erg = E_nu_erg / x
        Np_val = N_pro_interp_2d(E_p_erg, z_cm)
        if Np_val <= 0:
            integral_x.append(0.0)
            continue
            
        eps_min = 0.313 * ((m_p * c**2)**2) / (4 * E_p_erg)
        eps_max = 1e5 * eV_to_erg 
        if eps_min >= eps_max:
            integral_x.append(0.0)
            continue
            
        eps_grid = np.logspace(np.log10(eps_min), np.log10(eps_max), 70)
        integral_eps = []
        
        for eps in eps_grid:
            n_ph = n_ph_interp_obj(eps) * fator_diluicao_foton
            if n_ph <= 0:
                integral_eps.append(0.0)
                continue
            eta = 4 * eps * E_p_erg / ((m_p * c**2)**2)
            integral_eps.append(n_ph * (Phi_nu_Kelner(eta, x, 'numu') + Phi_nu_Kelner(eta, x, 'numu_bar')))
            
        int_eps_val = np.trapz(np.array(integral_eps) * eps_grid, x=np.log(eps_grid))
        integral_x.append(Np_val * int_eps_val)

    return np.trapz(np.array(integral_x), x=np.log(x_grid))

def suppression_factor(E_nu_prime_erg, B_prime, c, sigma_T):
    E_pi_prime, E_mu_prime = 4.0 * E_nu_prime_erg, 3.0 * E_nu_prime_erg
    t_dec_pi = (E_pi_prime / m_pi_erg) * tau_0_pi
    t_dec_mu = (E_mu_prime / m_mu_erg) * tau_0_mu
    
    rate_sync_pi = t_sync_rate(m_e, B_prime, E_pi_prime, c, sigma_T) * (m_e / (m_pi_erg/c**2))**4
    rate_sync_mu = t_sync_rate(m_e, B_prime, E_mu_prime, c, sigma_T) * (m_e / (m_mu_erg/c**2))**4
    
    t_sync_pi = 1.0 / rate_sync_pi if rate_sync_pi > 0 else np.inf
    t_sync_mu = 1.0 / rate_sync_mu if rate_sync_mu > 0 else np.inf
    
    S_pi = 1.0 / (1.0 + t_dec_pi / t_sync_pi)
    S_mu = 1.0 / (1.0 + t_dec_mu / t_sync_mu)
    return 0.5 * S_pi + 0.5 * (S_pi * S_mu)

def calculate_neutrino_flux_earth_pg_exact(E_nu_obs_GeV, base_p, N_pro_interp, n_ph_interp, E_max_pro):
    E_nu_obs_erg = E_nu_obs_GeV * gev_to_erg
    D_factor = 1.0 / (base_p['Gamma_b'] * (1.0 - base_p['beta'] * np.cos(base_p['theta_rad'])))
    E_nu_prime_erg = E_nu_obs_erg / D_factor 

    def volume_integrand(z_cm):
        Q_prime = calculate_Q_nu_pg_exact(E_nu_prime_erg, z_cm, N_pro_interp, n_ph_interp, base_p, E_max_pro)
        B_z = calculate_B_field(base_p['B_0_Gauss'], z_cm, base_p['z0'], base_p['m_idx'])
        B_prime_z = calculate_B_prime(B_z, base_p['Gamma_b'])
        
        Q_prime *= suppression_factor(E_nu_prime_erg, B_prime_z, c, sigma_T)
        area_secao = np.pi * (z_cm * np.tan(np.radians(base_p['xi_j_deg'])))**2 # Supondo expanção cônica baseada no código original
        return Q_prime * area_secao

    z_grid_total = np.linspace(base_p['Z_acc_cm'], base_p['Z_acc_cm'] + base_p['delta_z_cm'], 50)
    integrand_vals = np.array([volume_integrand(z) for z in z_grid_total])
    
    flux_differential = (D_factor / (d_CenA_cm**2)) * np.trapz(integrand_vals, x=z_grid_total) 
    return ((E_nu_obs_erg**2) * flux_differential) / gev_to_erg

In [ ]:
source_file = "source.txt"
sources_info = []

# Lemos o arquivo source original para resgatar os parâmetros que não foram para o CSV
with open(source_file, mode='r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        sources_info.append({
            'name': row['name'], 
            'xi_j': float(row['xi_j']) # Resgatando o ângulo direto da fonte!
        })

E_nu_GeV = np.logspace(-2, 7, 100)
all_fluxes = {}

print(f"=== INICIANDO INTEGRAÇÃO DE NEUTRINOS ({len(sources_info)} Fontes) ===")

for info in sources_info:
    name = info['name']
    print(f"\n-> Processando: {name}")
    
    try:
        # 1. Carrega os dados persistidos e INJETA o xi_j resgatado
        base_p = load_base_params(name)
        base_p['xi_j_deg'] = info['xi_j'] # Correção aplicada aqui!
        
        n_ph_interp = reconstruct_photon_interp(name)
        N_pro_interp, E_max_pro = reconstruct_proton_interp(name)
        
        # 2. Integração p-gamma (Kelner Exact)
        flux_pg = []
        for E in tqdm(E_nu_GeV, desc="   [p-gamma]", unit="pt"):
            flux_pg.append(calculate_neutrino_flux_earth_pg_exact(E, base_p, N_pro_interp, n_ph_interp, E_max_pro))
            
        # 3. Preparando grades de decaimento para pp
        print("   [pp] Construindo malhas de decaimento transitório...")
        E_grid_pi = np.logspace(np.log10(m_pi_erg), np.log10(E_max_pro * 20), 100)
        E_grid_mu = np.logspace(np.log10(m_mu_erg), np.log10(E_max_pro * 20), 100)
        z_end_jet = 100.0 * base_p['Z_acc_cm']
        z_grid_decay = np.logspace(np.log10(base_p['Z_acc_cm']), np.log10(z_end_jet), 100)
        
        def Q_pi_pp_wrapper(E, z): return Q_pi_pp_prime(E, z, base_p['n_c_prime'], N_pro_interp, E_max_pro)
        def Q_mu_L_wrapper(E, z): return Q_mu_prime(E, z, N_pi_pp_interp, E_max_pro, helicity='L')
        def Q_mu_R_wrapper(E, z): return Q_mu_prime(E, z, N_pi_pp_interp, E_max_pro, helicity='R')

        N_pi_pp_interp = build_N_decay_grid(E_grid_pi, z_grid_decay, b_total_z_pion, Q_pi_pp_wrapper, T_decay_pion, base_p, E_max_pro)
        N_mu_L_pp_interp = build_N_decay_grid(E_grid_mu, z_grid_decay, b_total_z_muon, Q_mu_L_wrapper, T_decay_muon, base_p, E_max_pro)
        N_mu_R_pp_interp = build_N_decay_grid(E_grid_mu, z_grid_decay, b_total_z_muon, Q_mu_R_wrapper, T_decay_muon, base_p, E_max_pro)

        # 4. Integração pp
        flux_pp = []
        for E in tqdm(E_nu_GeV, desc="   [pp] Integração de Fluxo", unit="pt"):
            flux_pp.append(calculate_neutrino_flux_earth_pp(E, base_p, N_pi_pp_interp, N_mu_L_pp_interp, N_mu_R_pp_interp, E_max_pro))
            
        all_fluxes[name] = {'pp': flux_pp, 'pg': flux_pg}
        
        # 5. Salva o fluxo final 
        filename = f"{DATA_DIR}{name}_neutrino_fluxes.csv"
        with open(filename, mode='w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['E_nu_GeV', 'flux_pp', 'flux_pg'])
            for i in range(len(E_nu_GeV)):
                writer.writerow([E_nu_GeV[i], flux_pp[i], flux_pg[i]])
                
        print(f"   === Fluxos salvos em {filename} ===")
        
    except Exception as e:
        print(f"   [ERRO] Falha ao processar a fonte {name}. Detalhes: {str(e)}")

print("\n=== CÁLCULO DE NEUTRINOS FINALIZADO ===")

=== INICIANDO INTEGRAÇÃO DE NEUTRINOS (1 Fontes) ===

-> Processando: Cen A


   [p-gamma]:  28%|███████████████████                                                 | 28/100 [04:11<11:48,  9.83s/pt]

In [ ]:
source_file = "source.txt"
sources = []
with open(source_file, mode='r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        sources.append(row['name'])

E_nu_GeV = np.logspace(-2, 7, 100) # Resolucão da curva de neutrinos
all_fluxes = {}

print(f"=== INICIANDO INTEGRAÇÃO DE NEUTRINOS ({len(sources)} Fontes) ===")

for name in sources:
    safe_name = name.replace(" ", "_").replace("–", "-")
    print(f"\n-> Processando: {name}")
    
    try:
        # 1. Carrega os dados persistidos do notebook anterior
        base_p = load_base_params(name)
        n_ph_interp = reconstruct_photon_interp(safe_name)
        N_pro_interp, E_max_pro = reconstruct_proton_interp(safe_name)
        
        # 2. Integração p-gamma (Kelner Exact)
        flux_pg = []
        for E in tqdm(E_nu_GeV, desc="   [p-gamma]", unit="pt"):
            flux_pg.append(calculate_neutrino_flux_earth_pg_exact(E, base_p, N_pro_interp, n_ph_interp, E_max_pro))
            
        # 3. Preparando grades de decaimento para pp
        print("   [pp] Construindo malhas de decaimento transitório...")
        E_grid_pi = np.logspace(np.log10(m_pi_erg), np.log10(E_max_pro * 20), 100)
        E_grid_mu = np.logspace(np.log10(m_mu_erg), np.log10(E_max_pro * 20), 100)
        z_end_jet = 100.0 * base_p['Z_acc_cm']
        z_grid_decay = np.logspace(np.log10(base_p['Z_acc_cm']), np.log10(z_end_jet), 100)
        
        def Q_pi_pp_wrapper(E, z): return Q_pi_pp_prime(E, z, base_p['n_c_prime'], N_pro_interp, E_max_pro)
        def Q_mu_L_wrapper(E, z): return Q_mu_prime(E, z, N_pi_pp_interp, E_max_pro, helicity='L')
        def Q_mu_R_wrapper(E, z): return Q_mu_prime(E, z, N_pi_pp_interp, E_max_pro, helicity='R')

        N_pi_pp_interp = build_N_decay_grid(E_grid_pi, z_grid_decay, b_total_z_pion, Q_pi_pp_wrapper, T_decay_pion, base_p, E_max_pro)
        N_mu_L_pp_interp = build_N_decay_grid(E_grid_mu, z_grid_decay, b_total_z_muon, Q_mu_L_wrapper, T_decay_muon, base_p, E_max_pro)
        N_mu_R_pp_interp = build_N_decay_grid(E_grid_mu, z_grid_decay, b_total_z_muon, Q_mu_R_wrapper, T_decay_muon, base_p, E_max_pro)

        # 4. Integração pp
        flux_pp = []
        for E in tqdm(E_nu_GeV, desc="   [pp] Integração de Fluxo", unit="pt"):
            flux_pp.append(calculate_neutrino_flux_earth_pp(E, base_p, N_pi_pp_interp, N_mu_L_pp_interp, N_mu_R_pp_interp, E_max_pro))
            
        all_fluxes[name] = {'pp': flux_pp, 'pg': flux_pg}
        
        # 5. Salva em CSV
        filename = f"{DATA_DIR}{safe_name}_neutrino_fluxes.csv"
        with open(filename, mode='w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['E_nu_GeV', 'flux_pp', 'flux_pg'])
            for i in range(len(E_nu_GeV)):
                writer.writerow([E_nu_GeV[i], flux_pp[i], flux_pg[i]])
                
        print(f"   === Fluxos salvos em {filename} ===")
        
    except Exception as e:
        print(f"   [ERRO] Falha ao processar a fonte {name}. Detalhes: {str(e)}")

print("\n=== CÁLCULO DE NEUTRINOS FINALIZADO ===")

In [ ]:
import matplotlib.cm as cm
import matplotlib.lines as mlines

plt.figure(figsize=(12, 8))
colores = cm.tab10(np.linspace(0, 1, len(all_fluxes)))

for idx, (name, fluxes) in enumerate(all_fluxes.items()):
    color = colores[idx]
    plt.loglog(E_nu_GeV, fluxes['pp'], color=color, linestyle='-', linewidth=2.5, label=f'{name} ($pp$)')
    plt.loglog(E_nu_GeV, fluxes['pg'], color=color, linestyle='--', linewidth=2.5, label=f'{name} ($p\gamma$)')

# ====================================================================
# 1. Curva de Sensibilidade do KM3NeT/ARCA (Hemisfério Sul)
# ====================================================================
E_km3_GeV = np.array([1e2, 1e3, 3e3, 1e4, 3e4, 1e5, 3e5, 1e6, 3e6, 1e7, 1e8])
Sens_km3 = np.array([1e-6, 4e-8, 1e-8, 3e-9, 1.2e-9, 8e-10, 8e-10, 1.2e-9, 2.5e-9, 8e-9, 4e-8])

interp_km3 = interp1d(np.log10(E_km3_GeV), np.log10(Sens_km3), kind='cubic', bounds_error=False, fill_value=np.nan)
E_plot_km3 = np.logspace(2, 7.5, 100)
Sens_plot_km3 = 10**interp_km3(np.log10(E_plot_km3))

plt.loglog(E_plot_km3, Sens_plot_km3, color='gray', linestyle='dotted', linewidth=2, label='KM3NeT/ARCA Sensitivity')

# ====================================================================
# 2. Curva de Sensibilidade do IceCube (Hemisfério Norte) - DADOS OFICIAIS
# ====================================================================
energia_nu_TeV = np.array([
    0.1, 0.316, 1.0, 3.16, 10.0, 
    31.6, 100.0, 316.2, 1000.0, 3162.2, 10000.0
])

sensibilidade_icecube_TeV = np.array([
    2.0e-11, 4.0e-12, 1.0e-12, 4.5e-13, 2.5e-13, 
    1.8e-13, 1.5e-13, 1.8e-13, 3.0e-13, 6.0e-13, 2.5e-12
])

# Convertendo X (TeV -> GeV) e Y (TeV cm^-2 s^-1 -> GeV cm^-2 s^-1)
E_ice_GeV = energia_nu_TeV * 1000
Sens_ice_GeV = sensibilidade_icecube_TeV * 1000

# Interpolação cúbica alinhada ao grid comum de plotagem
interp_ice = interp1d(np.log10(E_ice_GeV), np.log10(Sens_ice_GeV), kind='cubic', bounds_error=False, fill_value=np.nan)
Sens_plot_ice = 10**interp_ice(np.log10(E_plot_km3))

plt.loglog(E_plot_km3, Sens_plot_ice, color='black', linestyle='-.', linewidth=2, label='IceCube Sensitivity')

# ====================================================================
# Estilização do Gráfico Espectral
# ====================================================================
plt.xlabel(r'$E_\nu$ [GeV]', fontsize=15)
plt.ylabel(r'$E_\nu^2 d\Phi_\nu/dE_\nu$ [GeV cm$^{-2}$ s$^{-1}$]', fontsize=15)
plt.tick_params(axis='both', which='major', labelsize=13)

plt.grid(True, which='both', linestyle='--', alpha=0.4)
plt.ylim(1e-15, 1e-7) 
plt.xlim(100, 1e7)
plt.legend(frameon=False, fontsize=12)
plt.tight_layout()

# Salvamento físico do plot
os.makedirs("Image", exist_ok=True)
plot_filename = "Image/neutrino_flux_Population.png"
plt.savefig(plot_filename, dpi=200)
print(f"-> Plotagem salva com sucesso em '{plot_filename}'!")

plt.show()